In [44]:
import flax
from flax import nnx
print("Flax version:", flax.__version__)

import jax
import jax.numpy as jnp
import jax.tree_util as jtu
print("Jax version:", jax.__version__)

import numpy as np
from math import prod
from typing import Callable
from tqdm import tqdm

import torch
from torch.utils.data import TensorDataset, DataLoader, random_split
print(f"Pytorch version: {torch.__version__}")

import optax
print(f"Optax version: {optax.__version__}")

Flax version: 0.12.7
Jax version: 0.10.0
Pytorch version: 2.12.0
Optax version: 0.2.8


In [45]:
class NN_Build(nnx.Module):
    def __init__(
        self,
        input_shape: tuple[int, ...],
        output_dim: int,
        layer_dims: list,
        activation: Callable = nnx.relu,
        rngs: nnx.Rngs | None = None,
        ):
        if rngs is None:
            rngs = nnx.Rngs(0)

        self.input_shape = tuple(input_shape)
        self.output_dim = output_dim
        self.layer_dims = tuple(tuple(x) if not isinstance(x, tuple) else x for x in layer_dims)
        if len(self.layer_dims) == 0:
            raise ValueError("At least one layer dimension must be specified in layer_dims.")
        self.activation = activation
        self.rngs = rngs
        self.layers = nnx.List()

        self.check_layer_dims(self.input_shape, self.output_dim, self.layer_dims)

        if len(self.layer_dims) == 1:
            self._add_layer(self.layer_dims[0], None, self.layers, final_layer=True)
        else:
            self._add_layer(self.layer_dims[0], None, self.layers, final_layer=False)
            size_prev = self.layer_dims[0]
            for size_new in self.layer_dims[1:-1]:
                self._add_layer(size_new, size_prev, self.layers, final_layer=False)
                size_prev = size_new
            self._add_layer(self.layer_dims[-1], size_prev, self.layers, final_layer=True)
            size_prev = self.layer_dims[-1]

    def check_layer_dims(
        self,
        input_shape: tuple[int, ...],
        output_dim: int,
        layer_dims: list,
        ):
        data_shape = tuple(input_shape)

        for layer_dim in layer_dims:
            if len(layer_dim) not in (2, 3):
                raise ValueError(
                    f"Each layer dimension must be a tuple of length 2 (linear) or 3 "
                    f"(convolutional). Got {layer_dim}."
                )

            if len(layer_dim) == 2:
                current_flat_dim = prod(data_shape)
                if current_flat_dim != layer_dim[0]:
                    raise ValueError(
                        f"Expected input shape {data_shape} to be compatible with linear "
                        f"layer input dimension {layer_dim[0]}."
                    )
                data_shape = (layer_dim[1],)

            elif len(layer_dim) == 3:
                if len(data_shape) != 3:
                    raise ValueError(
                        f"Cannot add a convolutional layer after a linear layer, got "
                        f"layer dimensions {layer_dim} after input shape {data_shape}."
                    )

                if data_shape[2] != layer_dim[0]:
                    raise ValueError(
                        f"Expected input shape {data_shape} to be compatible with "
                        f"convolutional layer input channels {layer_dim[0]}."
                    )

                new_h = data_shape[0] - layer_dim[2] + 1
                new_w = data_shape[1] - layer_dim[2] + 1
                if new_h <= 0 or new_w <= 0:
                    raise ValueError(
                        f"Convolution with kernel size {layer_dim[2]} is too large for "
                        f"spatial shape {data_shape[:2]}."
                    )

                data_shape = (new_h, new_w, layer_dim[1])

        if data_shape != (output_dim,):
            raise ValueError(
                f"Final layer shape {data_shape} does not match output dimension {output_dim}."
            )
    class Flatten(nnx.Module):
        def __call__(self, x):
            return x.reshape(x.shape[:-3] + (-1,))

    def _add_layer(
        self,
        layer_dim: tuple,
        prev_layer_dim: tuple | None,
        layers: nnx.List,
        final_layer: bool = False,
        ):
        # class Flatten(nnx.Module):
        #     def __call__(self, x):
        #         return x.reshape((x.shape[0], -1))
        if prev_layer_dim is None:
            if len(layer_dim) == 2:
                if len(self.input_shape) == 1:
                    if self.input_shape[0] != layer_dim[0]:
                        raise ValueError(
                            f"Expected input shape {self.input_shape} to match linear "
                            f"input dimension {layer_dim[0]}."
                        )
                    layers.append(nnx.Linear(layer_dim[0], layer_dim[1], rngs=self.rngs))
                elif len(self.input_shape) == 3:
                    if prod(self.input_shape) != layer_dim[0]:
                        raise ValueError(
                            f"Expected flattened input size {prod(self.input_shape)} to "
                            f"match linear input dimension {layer_dim[0]}."
                        )
                    layers.append(self.Flatten())
                    layers.append(nnx.Linear(layer_dim[0], layer_dim[1], rngs=self.rngs))
                else:
                    raise ValueError(f"Unsupported input shape {self.input_shape}.")

            elif len(layer_dim) == 3:
                if len(self.input_shape) != 3:
                    raise ValueError(
                        f"Cannot add a convolutional layer after a linear layer, got "
                        f"layer dimensions {layer_dim} after input shape {self.input_shape}."
                    )
                if self.input_shape[2] != layer_dim[0]:
                    raise ValueError(
                        f"Expected input channels {self.input_shape[2]} to match "
                        f"convolutional input channels {layer_dim[0]}."
                    )
                layers.append(
                    nnx.Conv(
                        layer_dim[0],
                        layer_dim[1],
                        kernel_size=(layer_dim[2], layer_dim[2]),
                        padding="VALID",
                        rngs=self.rngs,
                    )
                )   

        else:
            if len(layer_dim) == 2 and len(prev_layer_dim) == 2:
                if prev_layer_dim[1] != layer_dim[0]:
                    raise ValueError(
                        f"Linear layer input dimension {layer_dim[0]} does not match "
                        f"previous layer output dimension {prev_layer_dim[1]}."
                    )
                layers.append(nnx.Linear(layer_dim[0], layer_dim[1], rngs=self.rngs))

            elif len(layer_dim) == 3 and len(prev_layer_dim) == 3:
                if prev_layer_dim[1] != layer_dim[0]:
                    raise ValueError(
                        f"Convolutional layer input channels {layer_dim[0]} do not match "
                        f"previous layer output channels {prev_layer_dim[1]}."
                    )
                layers.append(
                    nnx.Conv(
                        layer_dim[0],
                        layer_dim[1],
                        kernel_size=(layer_dim[2], layer_dim[2]),
                        padding="VALID",
                        rngs=self.rngs,
                    )
                )

            elif len(layer_dim) == 2 and len(prev_layer_dim) == 3:
                layers.append(self.Flatten())
                layers.append(nnx.Linear(layer_dim[0], layer_dim[1], rngs=self.rngs))

            elif len(layer_dim) == 3 and len(prev_layer_dim) == 2:
                raise ValueError(
                    f"Cannot add a convolutional layer after a linear layer, got "
                    f"layer dimensions {layer_dim} after {prev_layer_dim}."
                )

        if not final_layer:
            layers.append(self.activation)

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

In [46]:
input_shape = (60,60,5)
output_dim = 6
layer_dims = ((5,1,5) ,(56*56, 64), (64, 32), (32, output_dim))
CNN = NN_Build(input_shape, output_dim, layer_dims)
print(CNN)

NN_Build( # Param: 203,172 (812.7 KB), RngState: 2 (12 B), Total: 203,174 (812.7 KB)
  input_shape=(60, 60, 5),
  output_dim=6,
  layer_dims=((5, 1, 5), (3136, 64), (64, 32), (32, 6)),
  activation=<jax._src.custom_derivatives.custom_jvp object at 0x11401f770>,
  rngs=Rngs( # RngState: 2 (12 B)
    default=RngStream( # RngState: 2 (12 B)
      tag='default',
      key=RngKey( # 1 (8 B)
        value=Array((), dtype=key<fry>) overlaying:
        [0 0],
        tag='default'
      ),
      count=RngCount( # 1 (4 B)
        value=Array(8, dtype=uint32),
        tag='default'
      )
    )
  ),
  layers=List([
    Conv( # Param: 126 (504 B)
      kernel_shape=(5, 5, 5, 1),
      kernel=Param( # 125 (500 B)
        value=Array(shape=(5, 5, 5, 1), dtype=dtype('float32'))
      ),
      bias=Param( # 1 (4 B)
        value=Array([0.], dtype=float32)
      ),
      in_features=5,
      out_features=1,
      kernel_size=(5, 5),
      strides=1,
      padding='VALID',
      input_dilation=1,
    

In [47]:
with np.load('../sbi_lens_sims/mpi_job_4483820_combined.npz') as data:
    x = data['y']
    # Flax Conv layers expects data shape in (N, H, W, C) format
    y = data['theta']
    # Remove any NaN data points
    nan_mask = jnp.isnan(x).reshape(x.shape[0], -1).any(axis=1)
    clean_indices = jnp.where(~nan_mask)[0]
    x = x[clean_indices]
    y = y[clean_indices]
    print("x", x.shape, "\ny", y.shape)

print(CNN(x).shape)

def mse_loss(model, x, y):
    preds = model(x)
    if preds.shape != y.shape:
        raise ValueError(f"Output shpae of the model {preds.shape} does not match the shape of the labels {y.shape}")
    return jnp.mean((preds-y) ** 2)

loss = mse_loss(CNN, x, y)
print(loss)



x (4872, 60, 60, 5) 
y (4872, 6)
(4872, 6)
0.6135932


In [52]:
BATCH_SIZE = 64
LEARNING_RATE = 3e-4
EPOCHS = 150
STEPS = 15000
PRINT_EVERY = 3000
SEED = 5678
TRAIN_TEST_SPLIT = 0.8
rngs = nnx.Rngs(0)


x_tensor = torch.tensor(np.array(x), dtype=torch.float32)
y_tensor = torch.tensor(np.array(y), dtype=torch.float32)
dataset = TensorDataset(x_tensor, y_tensor)
train_size = int(TRAIN_TEST_SPLIT * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


optimizer = nnx.Optimizer(
  CNN, optax.adamw(LEARNING_RATE), wrt=nnx.Param
)
metrics = nnx.MultiMetric(
  loss=nnx.metrics.Average('loss'),
)
# nnx.display(optimizer)

@nnx.jit
def train_step(model, optimizer: nnx.Optimizer, metrics: nnx.MultiMetric, x_batch, y_batch, rngs: nnx.Rngs):
    """Train for a single step."""
    loss_value, grads = nnx.value_and_grad(mse_loss)(model, x_batch, y_batch)
    metrics.update(loss=loss_value)  # In-place updates.
    optimizer.update(model, grads)  # In-place updates.

@nnx.jit
def eval_step(model, metrics, x, y):
    """Calculate loss on test data without updating parameters."""
    loss_value = mse_loss(model, x, y)
    metrics.update(loss=loss_value)

 # Loop over our training dataset as many times as we need.
def infinite_trainloader():
    while True:
        yield from train_loader
    
for step, (x_batch, y_batch) in tqdm(zip(range(STEPS), infinite_trainloader())):
    train_step(CNN, optimizer, metrics, x_batch.numpy(), y_batch.numpy(), rngs)

    train_loss = metrics.compute()['loss']
    
    # --- EVALUATION PHASE ---
    if step % PRINT_EVERY == 0:
        metrics.reset() # Clear training metrics to track test metrics
        for batch_x, batch_y in test_loader:
            eval_step(CNN, metrics, batch_x.numpy(), batch_y.numpy())
        
        test_loss = metrics.compute()['loss']
        print(f"Epoch {epoch:3d} | Train Loss: {train_loss:.6f} | Test Loss: {test_loss:.6f}")

        
# for epoch in tqdm(range(EPOCHS)):
#     # --- TRAINING PHASE ---
#     metrics.reset() # Clear metrics for the new epoch
#     for x_batch, y_batch in train_loader:
#         train_step(CNN, optimizer, metrics, x_batch.numpy(), y_batch.numpy(), rngs)
    
#     train_loss = metrics.compute()['loss']
    
#     # --- EVALUATION PHASE ---
#     if epoch % PRINT_EVERY == 0:
#         metrics.reset() # Clear training metrics to track test metrics
#         for batch_x, batch_y in test_loader:
#             eval_step(CNN, metrics, batch_x.numpy(), batch_y.numpy())
        
#         test_loss = metrics.compute()['loss']
#         print(f"Epoch {epoch:3d} | Train Loss: {train_loss:.6f} | Test Loss: {test_loss:.6f}")
    # else:
        # Just print training progress on other epochs
        # print(f"Epoch {epoch:3d} | Train Loss: {train_loss:.6f}")

29it [00:00, 115.63it/s]

Epoch 149 | Train Loss: 0.009244 | Test Loss: 0.015576


3029it [00:25, 117.08it/s]

Epoch 149 | Train Loss: 0.003753 | Test Loss: 0.030302


6030it [00:51, 119.93it/s]

...

9029it [01:16, 124.45it/s]

Epoch 149 | Train Loss: 0.000660 | Test Loss: 0.039543


12030it [01:42, 119.60it/s]

Epoch 149 | Train Loss: 0.000493 | Test Loss: 0.041594


15000it [02:06, 118.16it/s]
